In [ ]:
import os
import sys
sys.path.insert(0,os.path.abspath('..'))

import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats.qmc import LatinHypercube

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)

MODELSDIR  = CONFIGS['filepaths']['models']
SPLITSDIR  = CONFIGS['filepaths']['splits']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
SRCONFIG   = CONFIGS['experiments']['sr']
NNSEEDS    = CONFIGS['experiments']['nn']['seeds']
TARGETVAR  = CONFIGS['domain']['target']

STATSFILE  = os.path.join(SPLITSDIR,'stats.json')
with open(STATSFILE,'r',encoding='utf-8') as f:
    STATS = json.load(f)
ZMIN = (0.0 - STATS[f'{TARGETVAR}_mean']) / STATS[f'{TARGETVAR}_std']

In [ ]:
SRFUNCTIONS = {
    'cube':lambda x:x**3, 'square':lambda x:x**2, 'neg':lambda x:-x,
    'sqrt':np.sqrt, 'exp':np.exp, 'log':np.log, 'abs':np.abs,
    'max':np.maximum, 'min':np.minimum}


def eval_form(form,x,predictornames,constants):
    ns = dict(SRFUNCTIONS,__builtins__={})
    for p in predictornames:
        ns[p] = x[p].values
    ns.update(constants)
    out = eval(form,ns)
    if np.ndim(out) == 0:
        out = np.full(len(x),float(out))
    return np.asarray(out,dtype=float)


def clipped_loss(raw,y,zmin):
    pred = zmin + np.maximum(raw,0.0)
    return float(np.mean((pred - y)**2))

In [ ]:
import xarray as xr
from scripts.models.sr.train import kernel_integrate

RUNCONFIG    = SRCONFIG['runs']['sr_atm']
FIELDVARS    = RUNCONFIG['fieldvars']
WEIGHTSFROM  = RUNCONFIG['weightsfrom']
FORM         = SRCONFIG['optimizedeqs']['sr_atm_eq']['form']
PREDICTORS   = FIELDVARS


def load_split(splitname):
    ds   = xr.open_dataset(os.path.join(SPLITSDIR,f'norm_{splitname}.h5'),engine='h5netcdf')
    ref  = ds[TARGETVAR].transpose('time','lat','lon')
    nsig = ds.sizes['sig']
    dsig = ds['dsig'].values
    fieldarrays = [ds[v].transpose('time','lat','lon','sig').values.reshape(-1,nsig) for v in FIELDVARS]
    fieldstack  = np.stack(fieldarrays,axis=1)
    surfmask    = ds['surfmask'].transpose('time','lat','lon','sig').values.reshape(-1,nsig) if 'surfmask' in ds else None
    seedfeatures = []
    for seed in NNSEEDS:
        wds = xr.open_dataset(os.path.join(WEIGHTSDIR,f'{WEIGHTSFROM}_{seed}_weights.nc'),engine='h5netcdf')
        seedfeatures.append(kernel_integrate(fieldstack,wds['k'].values,dsig,surfmask))
        wds.close()
    features = np.mean(seedfeatures,axis=0)
    columns = {v:features[:,i] for i,v in enumerate(FIELDVARS)}
    x = pd.DataFrame(columns)
    y = ref.values.ravel()
    validmask = np.isfinite(x).all(axis=1).values & np.isfinite(y)
    ds.close()
    return x,y,validmask


xtrain,ytrain,trainmask = load_split('train')
xvalid,yvalid,validmask = load_split('valid')

xfit = pd.concat([xtrain[trainmask],xvalid[validmask]]).reset_index(drop=True)
yfit = np.concatenate([ytrain[trainmask],yvalid[validmask]])

xval = xvalid[validmask][PREDICTORS].reset_index(drop=True)
yval = yvalid[validmask]

print(f'Fit samples: {len(yfit):,}')
print(f'Valid samples: {len(yval):,}')
print(f'Form: {FORM}')
print(f'zmin: {ZMIN:.6f}')

In [ ]:
BVALUES = np.linspace(0.5,1.5,41)
CURRENT = {'a':1.58,'b':1.0,'c':1.47,'d':0.37}
NRESTARTS = 20


def optimize_with_fixed_b(bval,xdata,ydata,zmin,nrestarts=NRESTARTS):
    freenames = ['a','c','d']
    sampler = LatinHypercube(d=len(freenames),seed=42)
    samples = sampler.random(n=nrestarts)

    def objective(params):
        constants = dict(zip(freenames,params),b=bval)
        raw = eval_form(FORM,xdata,PREDICTORS,constants)
        pred = zmin + np.maximum(raw,0.0)
        return float(np.mean((pred - ydata)**2))

    bestloss = np.inf
    bestparams = None
    inits = [np.array([CURRENT[c] for c in freenames])]
    for i in range(nrestarts - 1):
        init = np.array([CURRENT[c] + (samples[i,j]*6.0 - 3.0) for j,c in enumerate(freenames)])
        inits.append(init)
    for init in inits:
        res = minimize(objective,init,method='L-BFGS-B',
                       options={'maxiter':10000,'ftol':1e-14,'gtol':1e-10})
        if res.fun < bestloss:
            bestloss = res.fun
            bestparams = dict(zip(freenames,res.x),b=bval)
    return bestloss,bestparams


results = []
for i,bval in enumerate(BVALUES):
    loss,params = optimize_with_fixed_b(bval,xfit[PREDICTORS],yfit,ZMIN)
    valloss = clipped_loss(eval_form(FORM,xval,PREDICTORS,params),yval,ZMIN)
    results.append({'b':bval,'train_loss':loss,'valid_loss':valloss,
                    'a':params['a'],'c':params['c'],'d':params['d']})
    if i % 10 == 0:
        print(f'b={bval:.3f}: train={loss:.6f} valid={valloss:.6f} a={params["a"]:.3f} c={params["c"]:.3f} d={params["d"]:.3f}')

results = pd.DataFrame(results)
print(f'\nBest b by train loss: {results.loc[results["train_loss"].idxmin(),"b"]:.3f}')
print(f'Best b by valid loss: {results.loc[results["valid_loss"].idxmin(),"b"]:.3f}')

In [ ]:
fig,axes = plt.subplots(1,2,figsize=(12,4.5),constrained_layout=True)

ax = axes[0]
ax.plot(results['b'],results['train_loss'],'-o',ms=3,label='Train+Valid')
ax.plot(results['b'],results['valid_loss'],'-s',ms=3,label='Valid only')
ax.axvline(1.0,color='gray',ls='--',lw=0.8,label='b=1 (current)')
bestb_train = results.loc[results['train_loss'].idxmin(),'b']
bestb_valid = results.loc[results['valid_loss'].idxmin(),'b']
ax.axvline(bestb_train,color='C0',ls=':',lw=0.8,label=f'best train b={bestb_train:.2f}')
ax.axvline(bestb_valid,color='C1',ls=':',lw=0.8,label=f'best valid b={bestb_valid:.2f}')
ax.set_xlabel('b (thetae coefficient)')
ax.set_ylabel('MSE Loss')
ax.set_title('Loss vs fixed b')
ax.legend(fontsize=8)

ax = axes[1]
ax.plot(results['b'],results['a'],'-',label='a')
ax.plot(results['b'],results['c'],'-',label='c')
ax.plot(results['b'],results['d'],'-',label='d')
ax.axvline(1.0,color='gray',ls='--',lw=0.8)
ax.set_xlabel('b (thetae coefficient)')
ax.set_ylabel('Constant value')
ax.set_title('Other constants vs fixed b')
ax.legend(fontsize=8)

plt.savefig('optimize_b_sweep.png',dpi=150,bbox_inches='tight')
plt.show()

In [ ]:
from scripts.models.sr.optimize import multistart_optimize

print('Re-running full 4-parameter multistart optimization...')
print(f'Init: {CURRENT}')

constants,res = multistart_optimize(
    FORM,PREDICTORS,xfit[PREDICTORS],yfit,ZMIN,CURRENT,nworkers=1)

print(f'\nOptimized: {{", ".join(f"{k}={v:.4f}" for k,v in constants.items())}}')
print(f'Train+Valid loss: {res.fun:.6f}')
print(f'Converged: {res.success}')

valloss = clipped_loss(eval_form(FORM,xval,PREDICTORS,constants),yval,ZMIN)
print(f'Valid-only loss: {valloss:.6f}')
print(f'\nCompare to current: train={0.44474712:.6f} valid={0.45199465:.6f}')

In [ ]:
print('Numerical gradient of loss w.r.t. each constant at current optimum:')
eps = 1e-5
for cname in ['a','b','c','d']:
    cp = dict(CURRENT)
    cm = dict(CURRENT)
    cp[cname] = CURRENT[cname] + eps
    cm[cname] = CURRENT[cname] - eps
    lp = clipped_loss(eval_form(FORM,xfit[PREDICTORS],PREDICTORS,cp),yfit,ZMIN)
    lm = clipped_loss(eval_form(FORM,xfit[PREDICTORS],PREDICTORS,cm),yfit,ZMIN)
    grad = (lp - lm) / (2*eps)
    print(f'  d(loss)/d({cname}) = {grad:.8f}')